In [ ]:
# Cell 1: Install dependencies
!pip install -q transformers accelerate peft bitsandbytes flask pyngrok sentencepiece
print("Dependencies installed")

In [ ]:
# Cell 2: Set fixed LoRA adapter path (no search/extract)
import os

LORA_ADAPTER_PATH = "/kaggle/input/datasets/ranjaysingh07/llama-3-1-classification/meta_llama3-1_lora_adapter"

required_adapter_files = [
    "adapter_config.json",
    "adapter_model.safetensors",
]

print("Using fixed LoRA adapter path:")
print(f"  {LORA_ADAPTER_PATH}")

if not os.path.isdir(LORA_ADAPTER_PATH):
    raise FileNotFoundError(f"LoRA adapter directory not found: {LORA_ADAPTER_PATH}")

missing = [f for f in required_adapter_files if not os.path.exists(os.path.join(LORA_ADAPTER_PATH, f))]
if missing:
    raise FileNotFoundError(
        f"Missing required adapter file(s) in {LORA_ADAPTER_PATH}: {missing}"
    )

print("LoRA adapter path verified")

In [ ]:
# Cell 3: Load models (Llama 3.1 8B classification)

import os
import re
import torch
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

print("=" * 60)
print("LOADING MODELS")
print("=" * 60)

gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory before load: {torch.cuda.memory_allocated()/1e9:.1f} GB")

HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    for secret_name in ["HF_TOKEN", "huggingface_token", "HUGGING_FACE_TOKEN", "LLAMA_API"]:
        try:
            HF_TOKEN = user_secrets.get_secret(secret_name)
            if HF_TOKEN:
                print(f"HF token loaded from: {secret_name}")
                break
        except:
            pass
except:
    pass

if not HF_TOKEN:
    HF_TOKEN = os.environ.get("LLAMA_API")

if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)

# Fixed base model and adapter paths
CLASS_MODEL_ID = "meta-llama/Llama-3.1-8B"
GEN_MODEL_ID = CLASS_MODEL_ID
MODEL_SOURCE = CLASS_MODEL_ID

if not os.path.isdir(LORA_ADAPTER_PATH):
    raise FileNotFoundError(f"LoRA adapter directory not found: {LORA_ADAPTER_PATH}")
if not os.path.exists(os.path.join(LORA_ADAPTER_PATH, "adapter_config.json")):
    raise FileNotFoundError(f"adapter_config.json missing in: {LORA_ADAPTER_PATH}")
if not os.path.exists(os.path.join(LORA_ADAPTER_PATH, "adapter_model.safetensors")):
    raise FileNotFoundError(f"adapter_model.safetensors missing in: {LORA_ADAPTER_PATH}")

print(f"Classification model source: hub repo -> {MODEL_SOURCE}")
print(f"LoRA adapter source: {LORA_ADAPTER_PATH}")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

try:
    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_SOURCE,
        token=HF_TOKEN,
    )
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_SOURCE,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
        token=HF_TOKEN,
        low_cpu_mem_usage=True,
    )
except Exception as e:
    raise RuntimeError(
        "Failed to load gated Hub model meta-llama/Llama-3.1-8B. "
        "Accept the model license on Hugging Face and set a valid HF token in Kaggle secrets (HF_TOKEN)."
    ) from e

tokenizer.pad_token = tokenizer.eos_token
print(f"Base model loaded. GPU memory: {torch.cuda.memory_allocated()/1e9:.1f} GB")

print("Loading fixed LoRA adapter for classification...")
adapter_loaded = False
class_model = None

try:
    class_model = PeftModel.from_pretrained(base_model, LORA_ADAPTER_PATH)
    adapter_loaded = True
    print("LoRA adapter loaded")
    print(f"GPU memory after adapter: {torch.cuda.memory_allocated()/1e9:.1f} GB")
except Exception as e:
    raise RuntimeError(
        f"LoRA load failed from fixed path: {LORA_ADAPTER_PATH}. Error: {e}"
    ) from e

gen_model = class_model
gen_tokenizer = tokenizer
class_tokenizer = tokenizer
MODEL_ID = CLASS_MODEL_ID
class_model.eval()

print("=" * 60)
print("MODEL READY")
print(f"Classification model: {CLASS_MODEL_ID}")
print("LoRA status: LOADED")
print(f"GPU memory now: {torch.cuda.memory_allocated()/1e9:.1f} GB")
print("=" * 60)

In [ ]:

# Cell 4: Define Classification Categories (My 7 Categories)

# My 7 trained classification categories
CATEGORIES = [
    "SIMPLE INSTRUCTION",
    "INSTRUCTION WITH SEQUENCE",
    "PARALLEL INSTRUCTION",
    "INSTRUCTION WITH PURPOSE",
    "INSTRUCTION WITH REASON",
    "EXCLUSIVE INSTRUCTION (OBJECTS)",
    "EXCLUSIVE INSTRUCTION (ACTIONS)"
]

# Classification instruction (matching my training format exactly)
CLASSIFICATION_INSTRUCTION = """Classify the instruction type into one of: SIMPLE INSTRUCTION, INSTRUCTION WITH SEQUENCE, PARALLEL INSTRUCTION, INSTRUCTION WITH PURPOSE, INSTRUCTION WITH REASON, EXCLUSIVE INSTRUCTION (OBJECTS), EXCLUSIVE INSTRUCTION (ACTIONS)."""

# Category descriptions
CATEGORY_INFO = {
    "SIMPLE INSTRUCTION": "A basic, single action instruction",
    "INSTRUCTION WITH SEQUENCE": "Steps in order (then, after, first, next)",
    "PARALLEL INSTRUCTION": "Multiple simultaneous actions (and, while, simultaneously)",
    "INSTRUCTION WITH PURPOSE": "Action with goal (to, for, in order to, so that)",
    "INSTRUCTION WITH REASON": "Action with explanation (because, since, as)",
    "EXCLUSIVE INSTRUCTION (OBJECTS)": "Choice between objects (use X or Y)",
    "EXCLUSIVE INSTRUCTION (ACTIONS)": "Choice between actions (do X or do Y)"
}

print("My 7 Classification Categories:")
print("="*60)
for i, (cat, desc) in enumerate(CATEGORY_INFO.items(), 1):
    print(f"  {i}. {cat}")
    print(f"     → {desc}")
print("="*60)

In [ ]:
# Cell 5: Classification Function (Llama 3 8B)

# VERIFY LORA ADAPTER IS LOADED
print("Verifying LoRA adapter status...")

# Check if model has LoRA adapters
lora_loaded = False
try:
    if hasattr(class_model, 'peft_config'):
        print("LoRA adapter IS loaded!")
        print(f"   Config: {class_model.peft_config}")
        lora_loaded = True
    elif hasattr(class_model, 'active_adapter'):
        print(f"LoRA adapter active: {class_model.active_adapter}")
        lora_loaded = True
    else:
        print("WARNING: LoRA adapter NOT detected!")
        print("   The model will use base Llama 3 8B (untrained)")
        print("   Classification may be less accurate than your fine-tuned adapter.")
except Exception as e:
    print(f"Error checking LoRA: {e}")

print()


def classify_instruction(instruction, debug=False):
    """Classify an instruction using Llama 3 8B (and LoRA adapter if loaded)."""
    
    # Keep Alpaca-style prompt to stay compatible with prior training/inference behavior
    prompt = f"""### Instruction:
{CLASSIFICATION_INSTRUCTION}

### Input:
{instruction}

### Response:
"""
    
    inputs = class_tokenizer(prompt, return_tensors="pt").to(class_model.device)
    
    with torch.no_grad():
        outputs = class_model.generate(
            **inputs,
            max_new_tokens=30,
            do_sample=False,
            pad_token_id=class_tokenizer.eos_token_id,
            eos_token_id=class_tokenizer.eos_token_id
        )
    
    # Decode only the new tokens (exclude prompt)
    input_length = inputs['input_ids'].shape[1]
    new_tokens = outputs[0][input_length:]
    response = class_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    
    # Clean up response (take first line only)
    response = response.split("\n")[0].strip()
    response = response.strip(".")
    
    if debug:
        print(f"\n   DEBUG - Input: '{instruction}'")
        print(f"   DEBUG - Generated tokens: {new_tokens.tolist()[:20]}")
        print(f"   DEBUG - Raw response: '{response}'")
    
    # Direct match to categories (exact or partial)
    response_upper = response.upper()
    
    for cat in CATEGORIES:
        if cat in response_upper:
            if debug:
                print(f"   DEBUG - Matched: {cat}")
            return cat
    
    # Partial matching with key terms
    if "SEQUENCE" in response_upper:
        return "INSTRUCTION WITH SEQUENCE"
    if "PARALLEL" in response_upper:
        return "PARALLEL INSTRUCTION"
    if "PURPOSE" in response_upper:
        return "INSTRUCTION WITH PURPOSE"
    if "REASON" in response_upper:
        return "INSTRUCTION WITH REASON"
    if "EXCLUSIVE" in response_upper and "OBJECT" in response_upper:
        return "EXCLUSIVE INSTRUCTION (OBJECTS)"
    if "EXCLUSIVE" in response_upper and "ACTION" in response_upper:
        return "EXCLUSIVE INSTRUCTION (ACTIONS)"
    if "EXCLUSIVE" in response_upper:
        instr_lower = instruction.lower()
        if " or " in instr_lower:
            parts = instr_lower.split(" or ")
            action_words = ["walk", "run", "jump", "sit", "stand", "go", "come", "do", "make", "take"]
            if any(word in parts[0] for word in action_words):
                return "EXCLUSIVE INSTRUCTION (ACTIONS)"
            return "EXCLUSIVE INSTRUCTION (OBJECTS)"
    if "SIMPLE" in response_upper:
        return "SIMPLE INSTRUCTION"
    
    # Keyword-based fallback using instruction content
    instr_lower = instruction.lower()
    if any(word in instr_lower for word in ["then", "after", "first", "next", "before", "finally", "lastly"]):
        return "INSTRUCTION WITH SEQUENCE"
    if any(word in instr_lower for word in ["and", "while", "simultaneously", "together", "at the same time"]):
        if " and " in instr_lower:
            return "PARALLEL INSTRUCTION"
    if any(word in instr_lower for word in ["to ", "for ", "in order to", "so that", "so as to"]):
        return "INSTRUCTION WITH PURPOSE"
    if any(word in instr_lower for word in ["because", "since", "as ", "due to"]):
        return "INSTRUCTION WITH REASON"
    if " or " in instr_lower:
        parts = instr_lower.split(" or ")
        action_words = ["walk", "run", "jump", "sit", "stand", "go", "come", "do", "make", "take", "drink", "eat"]
        if any(word in parts[0] for word in action_words) or any(word in parts[1] for word in action_words):
            return "EXCLUSIVE INSTRUCTION (ACTIONS)"
        return "EXCLUSIVE INSTRUCTION (OBJECTS)"
    
    return "SIMPLE INSTRUCTION"

# DIAGNOSTIC TEST: Check if classifier is working
print("="*60)
print("DIAGNOSTIC TEST: Checking model behavior")
print("="*60)

test_from_training = [
    ("Boil water", "SIMPLE INSTRUCTION"),
    ("If water is not hot, heat it", "INSTRUCTION WITH REASON"),
    ("Pour hot water then stir", "INSTRUCTION WITH SEQUENCE"),
]

print("\n Testing with representative examples:")
print("-"*60)

for sentence, expected in test_from_training:
    predicted = classify_instruction(sentence, debug=True)
    match = "✓" if predicted == expected else "✗"
    print(f"\n  {match} '{sentence}'")
    print(f"      Expected: {expected}")
    print(f"      Got:      {predicted}")

print("\n\n" + "="*60)
print("Testing with broader examples:")
print("="*60)

test_cases = [
    ("Boil water", "SIMPLE INSTRUCTION"),
    ("Pour water then add salt", "INSTRUCTION WITH SEQUENCE"),
    ("Add sugar and milk together", "PARALLEL INSTRUCTION"),
    ("Heat it if you want to make it warm", "INSTRUCTION WITH PURPOSE"),
    ("Stir it because it will burn", "INSTRUCTION WITH REASON"),
    ("Use coffee or tea", "EXCLUSIVE INSTRUCTION (OBJECTS)"),
    ("Walk or run to the store", "EXCLUSIVE INSTRUCTION (ACTIONS)")
]

correct = 0
for sentence, expected in test_cases:
    predicted = classify_instruction(sentence, debug=True)
    match = "✓" if predicted == expected else "✗"
    if predicted == expected:
        correct += 1
    print(f"\n  {match} '{sentence}'")
    print(f"      Expected: {expected}")
    print(f"      Got:      {predicted}")

print(f"\n{'='*60}")
print(f"Accuracy: {correct}/{len(test_cases)} ({100*correct/len(test_cases):.0f}%)")
print(f"{'='*60}")

if correct < 3:
    print("\n" + "!"*60)
    print("LOW ACCURACY DETECTED!")
    print("!"*60)
    print("""
POSSIBLE CAUSES:

(1) LoRA adapter is missing or incompatible with Llama 3 8B
   - Ensure adapter was trained on the same Llama 3 base model
   - Ensure adapter files include adapter_config.json and adapter_model.safetensors

(2) Wrong checkpoint loaded
   - Make sure you are using the final or best checkpoint

(3) Prompt mismatch with training format
   - Verify your training template and keep inference prompt aligned
""")

In [ ]:
# Cell 6: Instruction generation (Gemini first, local Llama fallback)
import os
import re
import requests

def get_target_step_count(query, context_text=None, summary_mode=False):
    q = (query or '').lower()
    if summary_mode:
        return 8

    score = 10
    detail_terms = ["detailed", "complete", "comprehensive", "full", "step by step", "from scratch", "guide"]
    if any(t in q for t in detail_terms):
        score += 2

    domain_terms = ["install", "setup", "configure", "deploy", "recipe", "prepare", "application", "account", "gmail"]
    if any(t in q for t in domain_terms):
        score += 1

    token_count = len(re.findall(r"[a-zA-Z0-9]+", q))
    if token_count >= 8:
        score += 1

    if context_text:
        context_lines = [x.strip() for x in str(context_text).split("\n") if x.strip()]
        score += min(2, max(0, len(context_lines) // 8))

    return max(10, min(15, score))

def load_gemini_api_key():
    key_names = ["GEMINI_API_KEY", "GOOGLE_API_KEY", "GENAI_API_KEY"]
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        for name in key_names:
            try:
                value = user_secrets.get_secret(name)
                if value:
                    print(f"Gemini key loaded from: {name}")
                    return value
            except Exception:
                pass
    except Exception:
        pass

    for name in key_names:
        value = os.environ.get(name)
        if value:
            print(f"Gemini key loaded from env: {name}")
            return value

    print("Gemini key not found. Using local Llama fallback generation.")
    return None

GEMINI_API_KEY = load_gemini_api_key()
GEMINI_API_VERSIONS = ["v1beta", "v1"]
GEMINI_MODEL_PREFERENCES = [
    "gemini-2.5-flash-lite",
    "gemini-2.5-flash",
    "gemini-3-flash-lite",
    "gemini-3-flash",
    "gemini-2.0-flash-lite",
    "gemini-2.0-flash",
    "gemini-1.5-flash-8b",
    "gemini-1.5-flash",
    "gemini-1.5-pro",
]

ACTIVE_GEMINI_TARGET = None
DISCOVERED_MODELS_BY_VERSION = {}

def list_available_gemini_models(api_version):
    if not GEMINI_API_KEY:
        return []
    if api_version in DISCOVERED_MODELS_BY_VERSION:
        return DISCOVERED_MODELS_BY_VERSION[api_version]

    url = f"https://generativelanguage.googleapis.com/{api_version}/models?key={GEMINI_API_KEY}"
    models = []
    try:
        response = requests.get(url, timeout=40)
        if response.status_code != 200:
            DISCOVERED_MODELS_BY_VERSION[api_version] = []
            return []

        payload = response.json()
        for model_info in payload.get("models", []):
            methods = model_info.get("supportedGenerationMethods", []) or []
            if "generateContent" not in methods:
                continue
            name = model_info.get("name", "")
            if name.startswith("models/"):
                name = name.split("models/", 1)[1]
            if name.startswith("gemini") and name not in models:
                models.append(name)
    except Exception:
        models = []

    DISCOVERED_MODELS_BY_VERSION[api_version] = models
    return models

def build_model_targets():
    targets = []
    if ACTIVE_GEMINI_TARGET:
        targets.append(ACTIVE_GEMINI_TARGET)

    discovered_by_version = {
        version: list_available_gemini_models(version) for version in GEMINI_API_VERSIONS
    }

    for version in GEMINI_API_VERSIONS:
        available = discovered_by_version.get(version, [])
        for preferred in GEMINI_MODEL_PREFERENCES:
            if preferred in available:
                target = (version, preferred)
                if target not in targets:
                    targets.append(target)
        extra = [m for m in available if ("flash" in m.lower()) and m not in GEMINI_MODEL_PREFERENCES]
        for m in extra:
            target = (version, m)
            if target not in targets:
                targets.append(target)

    for version in GEMINI_API_VERSIONS:
        for preferred in GEMINI_MODEL_PREFERENCES:
            target = (version, preferred)
            if target not in targets:
                targets.append(target)

    return targets

def call_gemini(prompt, temperature=0.2, max_output_tokens=1200):
    global ACTIVE_GEMINI_TARGET
    if not GEMINI_API_KEY:
        return None

    targets = build_model_targets()
    last_error = None

    for api_version, model in targets:
        url = f"https://generativelanguage.googleapis.com/{api_version}/models/{model}:generateContent?key={GEMINI_API_KEY}"
        payload = {
            "contents": [{"parts": [{"text": prompt}]}],
            "generationConfig": {
                "temperature": temperature,
                "topP": 0.9,
                "topK": 40,
                "maxOutputTokens": max_output_tokens,
            },
        }

        try:
            response = requests.post(url, json=payload, timeout=90)
            if response.status_code == 200:
                data = response.json()
                candidates = data.get("candidates", [])
                if not candidates:
                    continue
                parts = candidates[0].get("content", {}).get("parts", [])
                text = "\n".join([p.get("text", "") for p in parts]).strip()
                if text:
                    if ACTIVE_GEMINI_TARGET != (api_version, model):
                        print(f"Gemini target selected: {model} ({api_version})")
                    ACTIVE_GEMINI_TARGET = (api_version, model)
                    return text
                continue
            if response.status_code == 404:
                continue
            last_error = f"{response.status_code} - {response.text[:300]}"
        except Exception as e:
            last_error = str(e)

    if last_error:
        print(f"Gemini API error after trying candidates: {last_error}")
    else:
        print("No compatible Gemini model available. Using local Llama fallback.")
    return None

def parse_numbered_steps(text):
    if not text:
        return []

    lines = [x.strip() for x in text.split('\n') if x.strip()]
    steps = []
    for line in lines:
        cleaned = re.sub(r'^(?:Step\s*)?\d+[\.\)\:\s-]+', '', line, flags=re.IGNORECASE).strip()
        cleaned = re.sub(r'^[\-\*]+\s*', '', cleaned).strip()
        cleaned = re.sub(r'\*+', '', cleaned).strip()
        cleaned = re.sub(r'\s+', ' ', cleaned)
        if cleaned and 3 <= len(cleaned) <= 140:
            steps.append(cleaned)

    if not steps:
        sentence_chunks = re.split(r'(?<=[.!?])\s+', re.sub(r'\s+', ' ', text))
        for s in sentence_chunks:
            s = s.strip(" -*\n\t")
            if 4 <= len(s) <= 140:
                steps.append(s)

    dedup = []
    seen = set()
    for s in steps:
        key = s.lower()[:130]
        if key not in seen:
            seen.add(key)
            dedup.append(s)
    return dedup

def compact_instruction(text):
    if not text:
        return ""
    t = re.sub(r'\s+', ' ', text).strip()
    t = re.sub(r'^[\-\*]+\s*', '', t)
    t = t.strip(" .;:,-")
    t = re.sub(r'\b(typically|generally|usually|carefully|recommended|desired)\b', '', t, flags=re.IGNORECASE)
    t = re.sub(r'\s+', ' ', t).strip(" .;:,-")
    words = t.split()
    if len(words) > 12:
        t = " ".join(words[:12]).strip(" .;:,-")
    if t:
        t = t[0].upper() + t[1:]
    return t

def postprocess_compact_steps(steps, max_steps):
    compacted = []
    seen = set()
    for raw in steps:
        short = compact_instruction(raw)
        if not short:
            continue
        key = short.lower()
        if key not in seen:
            seen.add(key)
            compacted.append(short)
        if len(compacted) >= max_steps:
            break
    return compacted

def build_gemini_prompt(query, target_steps, context_text=None, require_variety=False):
    context_block = ""
    if context_text:
        context_block = f"""
Website Content:
{context_text[:9000]}

STRICT WEBSITE RULES:
- Use ONLY website content facts.
- Keep wording faithful to website content.
- If info is missing, write: Information not found in website content.
"""

    variety_rules = ""
    if require_variety:
        variety_rules = """
Additional diversity rules:
- Use at least one line with 'then'.
- Use at least one line with 'and'.
- Use at least one line with 'or'.
- Use at least one line with 'if you want'.
- Use at least one line starting with 'If ...'.
"""

    return f"""You generate dataset-style instruction lines.

Task: Create exactly {target_steps} short instructions for this query.

Query:
{query}
{context_block}
Strict format rules:
1) Return only numbered lines (no heading, no explanation).
2) Keep each line short (3-12 words).
3) One direct action phrase per line.
4) Use simple plain language.
5) Do not output long paragraphs.
6) Use connectors where needed: then / and / or / if you want / If ...
{variety_rules}
Category style guide:
- SIMPLE: one direct action.
- PURPOSE: action + 'if you want ...'.
- EXCLUSIVE OBJECTS: same action + object A or object B.
- EXCLUSIVE ACTIONS: action A or action B.
- SEQUENCE: action A then action B.
- PARALLEL: action A and action B.
- REASON: If condition, action.

Output example:
1. Read the script carefully
2. Coordinate with director then brief actors
3. Prepare props and check lighting
"""

def generate_instructions_local_fallback(query, num_steps=8):
    prompt = f"""Task: Generate short dataset-style instructions.
Query: {query}
Return exactly {num_steps} numbered lines, each 3-12 words.
Use concise imperative style.
Steps:
1."""

    inputs = gen_tokenizer(prompt, return_tensors="pt").to(gen_model.device)
    with torch.no_grad():
        outputs = gen_model.generate(
            **inputs,
            max_new_tokens=450,
            do_sample=True,
            temperature=0.5,
            top_p=0.9,
            top_k=40,
            repetition_penalty=1.1,
            pad_token_id=gen_tokenizer.eos_token_id,
        )

    response = gen_tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "Steps:" in response:
        response = response.split("Steps:")[-1]
    parsed = parse_numbered_steps(response)
    return postprocess_compact_steps(parsed, num_steps)

def generate_instructions(query, num_steps=None, context_text=None, require_variety=False, summary_mode=False):
    print(f"Generating instructions for: '{query}'")

    if num_steps is None:
        num_steps = get_target_step_count(query, context_text=context_text, summary_mode=summary_mode)

    prompt = build_gemini_prompt(
        query,
        target_steps=num_steps,
        context_text=context_text,
        require_variety=require_variety,
    )
    gemini_output = call_gemini(prompt)

    if gemini_output:
        steps = parse_numbered_steps(gemini_output)
        steps = postprocess_compact_steps(steps, num_steps)
        if steps:
            print(f"Extracted {len(steps)} Gemini steps")
            return steps

    print("Using local Llama fallback generation")
    fallback_steps = generate_instructions_local_fallback(query, num_steps=num_steps)
    if fallback_steps:
        print(f"Extracted {len(fallback_steps)} local fallback steps")
        return fallback_steps

    return ["Create a clear instruction"]

In [ ]:
# Cell 7: Processing Functions
def rule_based_fallback_category(step_text):
    text = (step_text or "").strip()
    lower = text.lower()

    if "if you want" in lower or "in order to" in lower or "so that" in lower:
        return "INSTRUCTION WITH PURPOSE"
    if lower.startswith("if ") or re.search(r'\bif\b', lower):
        return "INSTRUCTION WITH REASON"
    if " then " in lower or any(k in lower for k in [" first ", " next ", " finally ", " after ", " before "]):
        return "INSTRUCTION WITH SEQUENCE"
    if " or " in lower:
        action_words = [
            "go", "walk", "run", "click", "open", "select", "choose", "use", "add", "mix",
            "write", "create", "install", "configure", "press", "stir", "heat", "cook", "boil",
            "serve", "garnish", "take", "make", "apply", "send", "call", "talk"
        ]
        parts = [p.strip() for p in lower.split(" or ") if p.strip()]
        if any(any(a in p for a in action_words) for p in parts):
            return "EXCLUSIVE INSTRUCTION (ACTIONS)"
        return "EXCLUSIVE INSTRUCTION (OBJECTS)"
    if " and " in lower:
        return "PARALLEL INSTRUCTION"
    return "SIMPLE INSTRUCTION"


def has_strong_pattern_match(step_text, category):
    lower = (step_text or "").lower()
    if category == "INSTRUCTION WITH PURPOSE":
        return any(k in lower for k in ["if you want", "in order to", "so that"])
    if category == "INSTRUCTION WITH REASON":
        return lower.startswith("if ") or (" if " in lower) or any(k in lower for k in ["because", "since", "due to"])
    if category == "INSTRUCTION WITH SEQUENCE":
        return " then " in lower or any(k in lower for k in [" first ", " next ", " finally ", " after ", " before "])
    if category == "PARALLEL INSTRUCTION":
        return " and " in lower
    if category.startswith("EXCLUSIVE INSTRUCTION"):
        return " or " in lower
    if category == "SIMPLE INSTRUCTION":
        return not any(k in lower for k in [" and ", " or ", " then ", " if ", "if you want", "because", "since"])
    return False


def classify_independently(step_text):
    try:
        category = classify_instruction(step_text)
        if category in CATEGORIES:
            strong = has_strong_pattern_match(step_text, category)
            confidence = 0.91 if strong else 0.78
            return {
                "category": category,
                "source": "lora-model",
                "model_confidence": round(confidence, 2)
            }
    except Exception as e:
        print(f"Model classification error, using fallback rules: {e}")

    category = rule_based_fallback_category(step_text)
    strong = has_strong_pattern_match(step_text, category)
    confidence = 0.66 if strong else 0.51
    return {
        "category": category,
        "source": "rule-fallback",
        "model_confidence": round(confidence, 2)
    }


def extend_steps_from_content(existing_steps, context_lines, target_steps):
    if len(existing_steps) >= target_steps:
        return existing_steps[:target_steps]

    expanded = list(existing_steps)
    seen = {s.lower().strip() for s in expanded if s}

    for line in context_lines:
        if len(expanded) >= target_steps:
            break
        clean = re.sub(r'\s+', ' ', line).strip(" .")
        if 4 <= len(clean) <= 120 and clean.lower() not in seen:
            expanded.append(clean)
            seen.add(clean.lower())

    if len(expanded) < target_steps:
        blob = " ".join(context_lines)
        chunks = re.split(r'(?<=[.!?])\s+', blob)
        for chunk in chunks:
            if len(expanded) >= target_steps:
                break
            clean = re.sub(r'\s+', ' ', chunk).strip(" .")
            if 5 <= len(clean) <= 120 and clean.lower() not in seen:
                expanded.append(clean)
                seen.add(clean.lower())

    return expanded[:target_steps]


def process_query(query):
    print("Mode: GENERATE from query")
    print(f"   Query: {query}")
    print("   Classifier: LoRA model (independent)")

    steps = generate_instructions(query, require_variety=True)
    print(f"   Generated {len(steps)} steps")

    results = []
    for i, step in enumerate(steps, 1):
        meta = classify_independently(step)
        category = meta["category"]
        item = {
            "step": i,
            "instruction": step,
            "category": category,
            "input": step,
            "output": category,
            "source": meta["source"],
            "model_confidence": meta["model_confidence"]
        }
        results.append(item)
        print(f"   {i}. [{category}] ({meta['source']} | conf={meta['model_confidence']}) {step[:80]}...")

    return {
        "mode": "generate",
        "query": query,
        "classifier": "lora-independent",
        "instructions": results
    }


def process_paragraph(paragraph):
    print("Mode: CLASSIFY paragraph")
    print(f"   Length: {len(paragraph)} chars")
    print("   Classifier: LoRA model (independent)")

    sentences = re.split(r'(?<=[.!?])\s+', paragraph.strip())
    results = []
    step_num = 1
    valid_sentences = [re.sub(r'\s+', ' ', s.strip()) for s in sentences if len(s.strip()) > 5]

    for sentence in valid_sentences:
        meta = classify_independently(sentence)
        category = meta["category"]
        results.append({
            "step": step_num,
            "instruction": sentence,
            "category": category,
            "input": sentence,
            "output": category,
            "source": meta["source"],
            "model_confidence": meta["model_confidence"]
        })
        print(f"   {step_num}. [{category}] ({meta['source']} | conf={meta['model_confidence']}) {sentence[:80]}...")
        step_num += 1

    return {
        "mode": "classify",
        "query": paragraph[:100] + "..." if len(paragraph) > 100 else paragraph,
        "classifier": "lora-independent",
        "instructions": results
    }


def is_summary_query(query):
    q = (query or "").lower()
    summary_terms = [
        "summarize", "summarise", "summerize", "summary", "overview", "key points",
        "main points", "what is this page", "website data", "web data",
        "give me my web", "extract data", "page data", "content from page"
    ]
    return any(term in q for term in summary_terms)


def query_tokens(query):
    q = re.sub(r'[^a-zA-Z0-9\s]', ' ', (query or '').lower())
    stop = {
        'the', 'is', 'a', 'an', 'to', 'for', 'and', 'or', 'of', 'on', 'in', 'with',
        'from', 'this', 'that', 'page', 'website', 'web', 'site', 'my', 'me', 'give',
        'according', 'based', 'please'
    }
    return [t for t in q.split() if len(t) > 2 and t not in stop]


def extract_ordered_instructions(content):
    lines = [re.sub(r'\s+', ' ', l).strip() for l in content.split('\n')]
    lines = [l for l in lines if l]

    numbered = []
    for line in lines:
        m = re.match(r'^\s*(\d{1,2})[\.)\-:]\s+(.*)$', line)
        if m:
            step_text = m.group(2).strip()
            if 6 <= len(step_text) <= 260:
                numbered.append(step_text)

    dedup = []
    seen = set()
    for x in numbered:
        k = x.lower()[:120]
        if k not in seen:
            seen.add(k)
            dedup.append(x)

    return dedup


def extract_page_snippets(content, max_items=12):
    if not content:
        return []

    lines = [re.sub(r'\s+', ' ', x).strip() for x in content.split('\n')]
    lines = [x for x in lines if 20 <= len(x) <= 280]

    noise_terms = [
        'http://', 'https://', 'www.', 'download article', 'last updated',
        'co-authored', 'fact checked', 'cookie', 'advertisement', 'subscribe',
        'comment', 'share this', 'privacy policy'
    ]
    lines = [x for x in lines if not any(t in x.lower() for t in noise_terms)]
    lines = [x for x in lines if not re.search(r'\b(thanks|thank you|hello|i live|can you help|dear)\b', x.lower())]

    if len(lines) < 4:
        sentence_chunks = re.split(r'(?<=[.!?])\s+', re.sub(r'\s+', ' ', content))
        lines.extend([x.strip() for x in sentence_chunks if 20 <= len(x.strip()) <= 280])

    deduped = []
    seen = set()
    for line in lines:
        key = line.lower()[:110]
        if key not in seen:
            seen.add(key)
            deduped.append(line)
        if len(deduped) >= max_items:
            break

    return deduped


def filter_by_query(snippets, query):
    tokens = query_tokens(query)
    if not tokens:
        return snippets

    scored = []
    for s in snippets:
        lower = s.lower()
        score = sum(1 for t in tokens if t in lower)
        if score > 0:
            scored.append((score, s))

    scored.sort(key=lambda x: x[0], reverse=True)
    if scored:
        return [x[1] for x in scored]
    return snippets


def process_website_content(content, query=None):
    print("Mode: WEBSITE content")
    print(f"   Content length: {len(content)} chars")
    print(f"   Query: {query if query else 'N/A'}")
    print("   Classifier: LoRA model (independent)")

    print("\n RAW WEBSITE CONTENT (first 1500 chars):")
    print("-" * 70)
    print(content[:1500])
    print("-" * 70)

    ordered_steps = extract_ordered_instructions(content)
    snippets = extract_page_snippets(content, max_items=22)
    snippets = filter_by_query(snippets, query)

    print("\n FILTERED WEBSITE CONTENT (first 12 lines):")
    print("-" * 70)
    for line in snippets[:12]:
        print(f"• {line}")
    print("-" * 70)

    q_tokens = query_tokens(query)
    page_blob = " ".join((ordered_steps[:25] + snippets[:25])).lower()
    overlap = sum(1 for t in q_tokens if t in page_blob)

    if q_tokens and overlap == 0:
        return {
            "mode": "website",
            "query": query or "Website Content Analysis",
            "summary": "Query topic not found in current webpage content. Disable page extraction or open a relevant page.",
            "classifier": "lora-independent",
            "instructions": []
        }

    if not snippets and not ordered_steps:
        return {
            "mode": "website",
            "query": query or "Website Content Analysis",
            "summary": "No usable content found on this page.",
            "classifier": "lora-independent",
            "instructions": []
        }

    context_lines = ordered_steps[:18] if ordered_steps else snippets[:18]
    context_text = "\n".join([f"- {x}" for x in context_lines])
    summary_mode = is_summary_query(query)
    if summary_mode:
        gemini_query = query or "Summarize this webpage professionally"
    else:
        gemini_query = query or "Extract complete professional instructions from this webpage"

    target_steps = get_target_step_count(gemini_query, context_text=context_text, summary_mode=summary_mode)
    structured_steps = generate_instructions(
        gemini_query,
        num_steps=target_steps,
        context_text=context_text,
        require_variety=False,
        summary_mode=summary_mode
    )
    structured_steps = extend_steps_from_content(structured_steps, context_lines, target_steps)

    results = []
    for i, step in enumerate(structured_steps[:15], 1):
        meta = classify_independently(step)
        category = meta["category"]
        item = {
            "step": i,
            "instruction": step,
            "category": category,
            "input": step,
            "output": category,
            "source": meta["source"],
            "model_confidence": meta["model_confidence"]
        }
        results.append(item)
        print(f"   {i}. [{category}] ({meta['source']} | conf={meta['model_confidence']}) {step[:80]}...")

    return {
        "mode": "website",
        "query": query or "Website Content Analysis",
        "summary": " ".join(snippets[:3]) if snippets else " ".join(structured_steps[:2]),
        "classifier": "lora-independent",
        "instructions": results
    }


def is_website_intent_query(query):
    q = (query or '').lower()
    website_terms = [
        'website', 'web site', 'webpage', 'web page', 'page content',
        'my page', 'this page', 'browser', 'chrome', 'site data', 'web data',
        'summarize my website', 'summarise my website', 'summerize my website', 'data of my webpage'
    ]
    return any(term in q for term in website_terms)


def process_request(data):
    query = data.get('query') or data.get('text')
    paragraph = data.get('paragraph')
    website_content = data.get('website_content') or data.get('pageContent')
    mode = data.get('mode', 'auto')

    if mode == 'auto':
        if paragraph:
            mode = 'classify'
        elif website_content or is_website_intent_query(query):
            mode = 'website'
        else:
            mode = 'generate'

    if mode == 'classify' and paragraph:
        return process_paragraph(paragraph)
    elif mode == 'website':
        if website_content:
            return process_website_content(website_content, query=query)
        return {
            "mode": "website",
            "query": query or "Website Content Analysis",
            "summary": "Website mode requested but no page content was received. Keep 'Extract relevant content from page' ON and reload the extension.",
            "classifier": "lora-independent",
            "instructions": []
        }
    elif query:
        return process_query(query)
    else:
        return {"error": "No valid input provided. Send query, paragraph, or website_content."}


print("✓ Processing functions ready")
print("✓ Classification in /parse uses LoRA independently with source/confidence fields")

In [ ]:
# Cell 7.1: Enhanced website chunking + Gemini relevance filtering (override)
import math

def split_text_into_word_chunks(text, min_words=1000, max_words=2000, overlap_words=120):
    words = re.findall(r"\S+", text or "")
    if not words:
        return []

    # For short pages, keep one chunk.
    if len(words) <= max_words:
        return [" ".join(words)]

    chunks = []
    start = 0
    step = max(1, max_words - overlap_words)
    while start < len(words):
        end = min(start + max_words, len(words))
        chunk_words = words[start:end]
        if len(chunk_words) >= min_words or end == len(words):
            chunks.append(" ".join(chunk_words))
        start += step
    return chunks

def quick_chunk_relevance_score(query, chunk):
    q_tokens = query_tokens(query)
    if not q_tokens:
        return 1.0
    lower_chunk = (chunk or "").lower()
    hit = sum(1 for t in q_tokens if t in lower_chunk)
    return hit / max(1, len(q_tokens))

def gemini_extract_query_relevant_steps(query, chunk_text, max_steps=8):
    prompt = f"""You are given webpage content and a user query.

User query:
{query}

Web content chunk:
{chunk_text[:14000]}

Task: Extract ONLY instructions relevant to the user query.
Rules:
1) Use only facts/actions present in the chunk.
2) If no relevant instructions are found, return exactly: NONE
3) Return numbered steps only, 1..N
4) Keep each step short (3-16 words).
5) Do not add unrelated tips.
6) Do not hallucinate or use outside knowledge.
"""
    out = call_gemini(prompt, temperature=0.0, max_output_tokens=900)
    if not out:
        return []
    if out.strip().upper() == "NONE":
        return []

    steps = parse_numbered_steps(out)
    return postprocess_compact_steps(steps, max_steps)

def gemini_filter_final_relevance(query, candidate_steps):
    if not candidate_steps:
        return []

    numbered = "\n".join([f"{i+1}. {s}" for i, s in enumerate(candidate_steps)])
    prompt = f"""User query: {query}

Candidate instructions:
{numbered}

Task: Keep only instructions relevant to the query.
Rules:
1) Return only the relevant lines as numbered list.
2) If none are relevant, return exactly: NONE
3) Do not rewrite meaning.
"""
    out = call_gemini(prompt, temperature=0.0, max_output_tokens=700)
    if not out:
        # Conservative fallback if Gemini is unavailable: lexical filter.
        q_tokens = query_tokens(query)
        if not q_tokens:
            return candidate_steps
        filtered = []
        for s in candidate_steps:
            lower = s.lower()
            if any(t in lower for t in q_tokens):
                filtered.append(s)
        return filtered

    if out.strip().upper() == "NONE":
        return []
    steps = parse_numbered_steps(out)
    # Keep only items that match original candidates to avoid drift.
    cand_set = {c.lower(): c for c in candidate_steps}
    kept = []
    seen = set()
    for s in steps:
        key = s.lower().strip()
        if key in cand_set and key not in seen:
            kept.append(cand_set[key])
            seen.add(key)
    return kept

def build_relevant_steps_from_website(query, content, target_steps=12):
    chunks = split_text_into_word_chunks(content, min_words=1000, max_words=2000, overlap_words=120)
    if not chunks:
        return []

    # Rank chunks by cheap lexical relevance first to reduce Gemini calls.
    ranked = sorted(
        [(quick_chunk_relevance_score(query, c), idx, c) for idx, c in enumerate(chunks)],
        key=lambda x: x[0],
        reverse=True
    )

    candidates = []
    for score, idx, chunk in ranked:
        if len(candidates) >= target_steps * 2:
            break
        # Skip very weak chunks when query terms exist.
        if query_tokens(query) and score <= 0:
            continue

        extracted = gemini_extract_query_relevant_steps(query, chunk, max_steps=min(8, target_steps))
        for s in extracted:
            if s not in candidates:
                candidates.append(s)
            if len(candidates) >= target_steps * 2:
                break

    # Final Gemini relevance pass.
    candidates = postprocess_compact_steps(candidates, target_steps * 2)
    final_steps = gemini_filter_final_relevance(query, candidates)
    final_steps = postprocess_compact_steps(final_steps, target_steps)

    # Fallback path if Gemini returns empty.
    if not final_steps:
        fallback = generate_instructions(
            query,
            num_steps=target_steps,
            context_text="\n".join([f"- {x}" for x in extract_page_snippets(content, max_items=20)]),
            require_variety=False,
            summary_mode=is_summary_query(query),
        )
        final_steps = postprocess_compact_steps(fallback, target_steps)

    return final_steps

# Override website processing to enforce query relevance before classification.
def process_website_content(content, query=None):
    print("Mode: WEBSITE content (enhanced relevance)")
    print(f"   Content length: {len(content)} chars")
    print(f"   Query: {query if query else 'N/A'}")
    print("   Classifier: LoRA model (independent)")

    if not content or not str(content).strip():
        return {
            "mode": "website",
            "query": query or "Website Content Analysis",
            "summary": "No usable content found on this page.",
            "classifier": "lora-independent",
            "instructions": []
        }

    effective_query = query or "Extract complete professional instructions from this webpage"
    target_steps = get_target_step_count(effective_query, context_text=content[:4000], summary_mode=is_summary_query(effective_query))
    target_steps = max(8, min(15, target_steps))

    # Build relevant instructions from chunked content with Gemini filtering.
    relevant_steps = build_relevant_steps_from_website(effective_query, content, target_steps=target_steps)

    # Final safety lexical filter for strict relevance.
    q_tokens = query_tokens(effective_query)
    if q_tokens:
        safe_steps = []
        for s in relevant_steps:
            lower = s.lower()
            if any(t in lower for t in q_tokens):
                safe_steps.append(s)
        if safe_steps:
            relevant_steps = safe_steps

    results = []
    for i, step in enumerate(relevant_steps[:15], 1):
        meta = classify_independently(step)
        category = meta["category"]
        item = {
            "step": i,
            "instruction": step,
            "category": category,
            "input": step,
            "output": category,
            "source": meta["source"],
            "model_confidence": meta["model_confidence"]
        }
        results.append(item)
        print(f"   {i}. [{category}] ({meta['source']} | conf={meta['model_confidence']}) {step[:90]}...")

    if not results:
        return {
            "mode": "website",
            "query": effective_query,
            "summary": "No query-relevant instructions found in webpage content.",
            "classifier": "lora-independent",
            "instructions": []
        }

    summary_text = " ".join([x["instruction"] for x in results[:3]])
    return {
        "mode": "website",
        "query": effective_query,
        "summary": summary_text,
        "classifier": "lora-independent",
        "instructions": results
    }

def process_request(data):
    query = data.get('query') or data.get('text')
    paragraph = data.get('paragraph')
    website_content = data.get('website_content') or data.get('pageContent')
    instruction = data.get('instruction') or data.get('text') or data.get('query')
    mode = (data.get('mode') or 'auto').lower()

    # Explicit modes from Chrome extension UI
    if mode == 'website':
        if website_content:
            return process_website_content(website_content, query=query)
        return {
            'mode': 'website',
            'query': query or 'Website Content Analysis',
            'summary': "Website mode selected but no page content was received.",
            'classifier': 'lora-independent',
            'instructions': []
        }

    if mode == 'generate':
        if query:
            return process_query(query)
        return {'error': 'No query provided for generate mode.'}

    if mode in ('classify-single', 'raw-classify', 'classify_instruction'):
        if not instruction:
            return {'error': 'No instruction provided for classify-single mode.'}
        meta = classify_independently(instruction)
        return {
            'mode': 'classify-single',
            'query': instruction,
            'instruction': instruction,
            'category': meta['category'],
            'source': meta['source'],
            'model_confidence': meta['model_confidence']
        }

    # Backward-compatible auto behavior
    if mode == 'auto':
        if paragraph and not website_content:
            return process_paragraph(paragraph)
        if website_content or is_website_intent_query(query):
            if website_content:
                return process_website_content(website_content, query=query)
            return {
                'mode': 'website',
                'query': query or 'Website Content Analysis',
                'summary': "Website mode requested but no page content was received.",
                'classifier': 'lora-independent',
                'instructions': []
            }
        if query:
            return process_query(query)

    # Existing classify paragraph mode
    if mode == 'classify' and paragraph:
        return process_paragraph(paragraph)

    if query:
        return process_query(query)

    return {'error': 'No valid input provided. Send query, paragraph, instruction, or website content.'}

print("✓ Enhanced website chunking + relevance filtering enabled")
print("✓ Only query-relevant instructions are passed to QLoRA classifier")
print("✓ process_request override supports modes: website | generate | classify-single")

In [ ]:
# Cell 7.2: Backend visibility + real confidence override
import torch
import re

def _print_block(title, text, max_chars=2500):
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)
    text = "" if text is None else str(text)
    if len(text) > max_chars:
        print(text[:max_chars] + f"\n... [truncated {len(text)-max_chars} chars]")
    else:
        print(text)
    print("=" * 80)

def _safe_float(x, default=0.0):
    try:
        return float(x)
    except Exception:
        return default

def _label_logprobs_for_instruction(instruction):
    """Compute normalized category probabilities from the model (real confidence proxy)."""
    prompt = f"""### Instruction:
{CLASSIFICATION_INSTRUCTION}

### Input:
{instruction}

### Response:
"""
    
    inputs = class_tokenizer(prompt, return_tensors="pt").to(class_model.device)
    input_ids = inputs["input_ids"]
    attn_mask = inputs.get("attention_mask", torch.ones_like(input_ids))

    logps = []
    with torch.no_grad():
        for cat in CATEGORIES:
            target_text = " " + cat
            target_ids = class_tokenizer(
                target_text,
                return_tensors="pt",
                add_special_tokens=False,
            )["input_ids"].to(class_model.device)

            full_ids = torch.cat([input_ids, target_ids], dim=1)
            full_mask = torch.cat([attn_mask, torch.ones_like(target_ids)], dim=1)

            out = class_model(input_ids=full_ids, attention_mask=full_mask)
            logits = out.logits[0]

            start = input_ids.shape[1]
            total_lp = 0.0
            for i in range(target_ids.shape[1]):
                pos = start + i - 1
                tok_id = target_ids[0, i]
                token_lp = torch.log_softmax(logits[pos], dim=-1)[tok_id].item()
                total_lp += token_lp
            logps.append(total_lp)

    probs = torch.softmax(torch.tensor(logps, dtype=torch.float32), dim=0).tolist()
    return {cat: _safe_float(p) for cat, p in zip(CATEGORIES, probs)}

def classify_independently(step_text):
    """Override: classification + real confidence from model label probabilities."""
    try:
        category = classify_instruction(step_text)
        if category not in CATEGORIES:
            category = rule_based_fallback_category(step_text)

        probs = _label_logprobs_for_instruction(step_text)
        conf = probs.get(category, 0.0)
        sorted_probs = sorted(probs.items(), key=lambda x: x[1], reverse=True)
        top2_gap = 0.0
        if len(sorted_probs) >= 2:
            top2_gap = sorted_probs[0][1] - sorted_probs[1][1]

        return {
            "category": category,
            "source": "lora-model",
            "model_confidence": round(conf, 4),
            "top2_gap": round(top2_gap, 4),
            "probabilities": probs,
        }
    except Exception as e:
        print(f"Model confidence scoring failed, fallback rules used: {e}")
        category = rule_based_fallback_category(step_text)
        strong = has_strong_pattern_match(step_text, category)
        confidence = 0.66 if strong else 0.51
        return {
            "category": category,
            "source": "rule-fallback",
            "model_confidence": round(confidence, 4),
            "top2_gap": 0.0,
            "probabilities": {},
        }

def process_query(query):
    print("Mode: GENERATE from query (verbose)")
    _print_block("USER QUERY", query, max_chars=1200)

    steps = generate_instructions(query, require_variety=True)
    _print_block("GEMINI/LLM GENERATED INSTRUCTIONS", "\n".join([f"{i+1}. {s}" for i, s in enumerate(steps)]), max_chars=5000)

    results = []
    for i, step in enumerate(steps, 1):
        meta = classify_independently(step)
        category = meta["category"]
        item = {
            "step": i,
            "instruction": step,
            "category": category,
            "input": step,
            "output": category,
            "source": meta["source"],
            "model_confidence": meta["model_confidence"],
            "top2_gap": meta.get("top2_gap", 0.0),
            "label_probabilities": meta.get("probabilities", {}),
        }
        results.append(item)
        print(f"   {i}. [{category}] conf={meta['model_confidence']} gap={meta.get('top2_gap', 0.0)}")

    return {
        "mode": "generate",
        "query": query,
        "classifier": "lora-independent",
        "instructions": results
    }

def process_paragraph(paragraph):
    print("Mode: CLASSIFY paragraph/raw instruction (verbose)")
    _print_block("USER GIVEN RAW INPUT", paragraph, max_chars=4000)

    sentences = re.split(r'(?<=[.!?])\s+', paragraph.strip())
    results = []
    step_num = 1
    valid_sentences = [re.sub(r'\s+', ' ', s.strip()) for s in sentences if len(s.strip()) > 0]

    for sentence in valid_sentences:
        meta = classify_independently(sentence)
        category = meta["category"]
        results.append({
            "step": step_num,
            "instruction": sentence,
            "category": category,
            "input": sentence,
            "output": category,
            "source": meta["source"],
            "model_confidence": meta["model_confidence"],
            "top2_gap": meta.get("top2_gap", 0.0),
            "label_probabilities": meta.get("probabilities", {}),
        })
        print(f"   {step_num}. [{category}] conf={meta['model_confidence']} gap={meta.get('top2_gap', 0.0)}")
        step_num += 1

    return {
        "mode": "classify",
        "query": paragraph[:100] + "..." if len(paragraph) > 100 else paragraph,
        "classifier": "lora-independent",
        "instructions": results
    }

def process_website_content(content, query=None):
    print("Mode: WEBSITE content (verbose enhanced)")
    effective_query = query or "Extract complete professional instructions from this webpage"

    _print_block("RAW WEB CONTENT RECEIVED", content, max_chars=6000)
    _print_block("USER QUERY", effective_query, max_chars=1500)

    target_steps = get_target_step_count(
        effective_query,
        context_text=content[:4000],
        summary_mode=is_summary_query(effective_query),
    )
    target_steps = max(8, min(15, target_steps))

    relevant_steps = build_relevant_steps_from_website(effective_query, content, target_steps=target_steps)
    _print_block(
        "FILTERED INSTRUCTIONS FROM WEB CONTENT (BY GEMINI)",
        "\n".join([f"{i+1}. {s}" for i, s in enumerate(relevant_steps)]) if relevant_steps else "NONE",
        max_chars=6000,
    )

    results = []
    for i, step in enumerate(relevant_steps[:15], 1):
        meta = classify_independently(step)
        category = meta["category"]
        item = {
            "step": i,
            "instruction": step,
            "category": category,
            "input": step,
            "output": category,
            "source": meta["source"],
            "model_confidence": meta["model_confidence"],
            "top2_gap": meta.get("top2_gap", 0.0),
            "label_probabilities": meta.get("probabilities", {}),
        }
        results.append(item)
        print(f"   {i}. [{category}] conf={meta['model_confidence']} gap={meta.get('top2_gap', 0.0)}")

    if not results:
        return {
            "mode": "website",
            "query": effective_query,
            "summary": "No query-relevant instructions found in webpage content.",
            "classifier": "lora-independent",
            "instructions": []
        }

    summary_text = " ".join([x["instruction"] for x in results[:3]])
    return {
        "mode": "website",
        "query": effective_query,
        "summary": summary_text,
        "classifier": "lora-independent",
        "instructions": results
    }

def process_request(data):
    query = data.get('query') or data.get('text')
    paragraph = data.get('paragraph')
    website_content = data.get('website_content') or data.get('pageContent')
    instruction = data.get('instruction') or data.get('text') or data.get('query')
    mode = (data.get('mode') or 'auto').lower()

    _print_block("REQUEST MODE", mode, max_chars=400)

    if mode == 'website':
        if website_content:
            return process_website_content(website_content, query=query)
        return {
            'mode': 'website',
            'query': query or 'Website Content Analysis',
            'summary': 'Website mode selected but no page content was received.',
            'classifier': 'lora-independent',
            'instructions': []
        }

    if mode == 'generate':
        if query:
            return process_query(query)
        return {'error': 'No query provided for generate mode.'}

    if mode in ('classify-single', 'raw-classify', 'classify_instruction'):
        if not instruction:
            return {'error': 'No instruction provided for classify-single mode.'}
        _print_block("USER GIVEN INSTRUCTION (RAW CLASSIFY)", instruction, max_chars=2000)
        meta = classify_independently(instruction)
        return {
            'mode': 'classify-single',
            'query': instruction,
            'instruction': instruction,
            'category': meta['category'],
            'source': meta['source'],
            'model_confidence': meta['model_confidence'],
            'top2_gap': meta.get('top2_gap', 0.0),
            'label_probabilities': meta.get('probabilities', {}),
        }

    # Backward-compatible auto behavior
    if mode == 'auto':
        if paragraph and not website_content:
            return process_paragraph(paragraph)
        if website_content or is_website_intent_query(query):
            if website_content:
                return process_website_content(website_content, query=query)
            return {
                'mode': 'website',
                'query': query or 'Website Content Analysis',
                'summary': 'Website mode requested but no page content was received.',
                'classifier': 'lora-independent',
                'instructions': []
            }
        if query:
            return process_query(query)

    if mode == 'classify' and paragraph:
        return process_paragraph(paragraph)

    if query:
        return process_query(query)

    return {'error': 'No valid input provided. Send query, paragraph, instruction, or website content.'}

print("✓ Verbose backend logging enabled")
print("✓ Real-confidence scoring enabled for LoRA classification")
print("✓ Kaggle logs will show: raw web content, Gemini-filtered steps, generated steps, user input, confidence")

In [ ]:
# Cell 7.3: Realistic generation + LoRA-only classification override
import re
import math

def extract_requested_step_count(query):
    """Read explicit user request like '20 steps' or 'give me 15 instructions'."""
    q = (query or "").lower()

    patterns = [
        r"\b(\d{1,2})\s*(?:steps?|instructions?|points?|items?)\b",
        r"\b(?:give|create|generate|make|return|provide)\s*(\d{1,2})\b",
        r"\b(?:at least|min(?:imum)?)\s*(\d{1,2})\b",
    ]

    for pat in patterns:
        m = re.search(pat, q)
        if m:
            n = int(m.group(1))
            return max(6, min(30, n))
    return None

def get_target_step_count(query, context_text=None, summary_mode=False):
    q = (query or "").lower()

    explicit = extract_requested_step_count(q)
    if explicit is not None:
        return explicit

    if summary_mode:
        return 10

    score = 10

    short_terms = ["short", "brief", "quick", "concise"]
    if any(t in q for t in short_terms):
        score -= 2

    detail_terms = [
        "detailed", "complete", "comprehensive", "full", "step by step",
        "from scratch", "guide", "deep", "end to end", "realistic"
    ]
    if any(t in q for t in detail_terms):
        score += 4

    complex_terms = [
        "deploy", "architecture", "production", "pipeline", "integration",
        "security", "setup", "configure", "troubleshoot", "migration"
    ]
    if any(t in q for t in complex_terms):
        score += 3

    token_count = len(re.findall(r"[a-zA-Z0-9]+", q))
    if token_count >= 12:
        score += 2
    if token_count >= 22:
        score += 2
    if token_count >= 40:
        score += 2

    if context_text:
        context_lines = [x.strip() for x in str(context_text).split("\n") if x.strip()]
        score += min(4, max(0, len(context_lines) // 8))

    return max(8, min(26, score))

def _merge_unique_steps(base_steps, new_steps, max_steps):
    merged = []
    seen = set()
    for s in (base_steps or []) + (new_steps or []):
        if not s:
            continue
        clean = compact_instruction(s)
        if not clean:
            continue
        key = clean.lower()
        if key in seen:
            continue
        seen.add(key)
        merged.append(clean)
        if len(merged) >= max_steps:
            break
    return merged

def generate_instructions(query, num_steps=None, context_text=None, require_variety=False, summary_mode=False):
    print(f"Generating realistic instructions for: '{query}'")

    if num_steps is None:
        num_steps = get_target_step_count(query, context_text=context_text, summary_mode=summary_mode)

    prompt = build_gemini_prompt(
        query,
        target_steps=num_steps,
        context_text=context_text,
        require_variety=require_variety,
    )
    gemini_output = call_gemini(prompt, temperature=0.25, max_output_tokens=1800)

    steps = []
    if gemini_output:
        parsed = parse_numbered_steps(gemini_output)
        steps = postprocess_compact_steps(parsed, num_steps)
        print(f"Gemini first pass steps: {len(steps)}/{num_steps}")

    if len(steps) < num_steps:
        remaining = num_steps - len(steps)
        if gemini_output:
            continuation_prompt = f"""Continue the same task and return ONLY the remaining {remaining} numbered instructions.
Existing instructions (do not repeat):
{"".join([f"\n{i+1}. {s}" for i, s in enumerate(steps)])}

User query: {query}
Rules:
1) Return numbered lines only.
2) Keep each line practical and concrete.
3) No duplicates from existing instructions.
"""
            cont_out = call_gemini(continuation_prompt, temperature=0.25, max_output_tokens=1200)
            cont_steps = postprocess_compact_steps(parse_numbered_steps(cont_out or ""), remaining)
            steps = _merge_unique_steps(steps, cont_steps, num_steps)
            print(f"Gemini continuation added: {len(cont_steps)}; total={len(steps)}/{num_steps}")

    if len(steps) < num_steps:
        fallback_needed = num_steps - len(steps)
        print(f"Using local Llama fallback for remaining {fallback_needed} steps")
        fallback_steps = generate_instructions_local_fallback(query, num_steps=max(8, fallback_needed + 4))
        steps = _merge_unique_steps(steps, fallback_steps, num_steps)

    if not steps:
        return ["Start by gathering the required information"]

    print(f"Final generated steps: {len(steps)}")
    return steps[:num_steps]

def classify_independently(step_text):
    """LoRA-only classification using model probabilities. No rule-based fallback."""
    probs = _label_logprobs_for_instruction(step_text)
    if not probs:
        raise RuntimeError("LoRA probability scoring returned empty output.")

    ranked = sorted(probs.items(), key=lambda x: x[1], reverse=True)
    top_label, top_prob = ranked[0]
    top2_gap = top_prob - ranked[1][1] if len(ranked) > 1 else top_prob

    return {
        "category": top_label,
        "source": "lora-model-only",
        "model_confidence": round(float(top_prob), 4),
        "top2_gap": round(float(top2_gap), 4),
        "probabilities": probs,
    }

print("✓ Realistic generation override enabled")
print("✓ LLM query mode now supports variable/explicit step counts + continuation")
print("✓ Classification is now LoRA-model-only (no rule-based fallback)")

In [ ]:
# Cell 7.4: Raw classify multi-sentence override (split by full stop)
import re

_previous_process_request = process_request

def _split_raw_classify_units(text):
    """Split raw classify input into sentence/list units and keep order."""
    text = "" if text is None else str(text)
    text = re.sub(r"\s+", " ", text).strip()
    if not text:
        return []

    # First split numbered list items if present (e.g., "1. ... 2. ...").
    numbered_parts = re.split(r"(?=\b\d{1,2}[\.)]\s+)", text)
    chunks = []
    if len([p for p in numbered_parts if p.strip()]) > 1:
        chunks = [re.sub(r"^\s*\d{1,2}[\.)]\s*", "", p).strip() for p in numbered_parts if p.strip()]
    else:
        chunks = [text]

    # Then split each chunk by sentence boundary, including no-space cases like "tea.In".
    units = []
    for chunk in chunks:
        parts = re.split(r"(?<=[.!?])\s+|(?<=[.!?])(?=[A-Z])|\n+", chunk)
        for p in parts:
            s = re.sub(r"\s+", " ", p).strip().strip(".;")
            if s:
                units.append(s)

    # Deduplicate while preserving order.
    out = []
    seen = set()
    for u in units:
        key = u.lower()
        if key not in seen:
            seen.add(key)
            out.append(u)
    return out

def process_request(data):
    mode = (data.get('mode') or 'auto').lower().strip()

    if mode in ('classify-single', 'raw-classify', 'raw classify', 'classify_instruction'):
        instruction = data.get('instruction') or data.get('text') or data.get('query')
        if not instruction:
            return {'error': 'No instruction provided for classify-single mode.'}

        _print_block("USER GIVEN INSTRUCTION (RAW CLASSIFY)", instruction, max_chars=3000)
        units = _split_raw_classify_units(instruction)

        if not units:
            return {'error': 'No valid instruction text found after splitting.'}

        # Backward compatibility: single sentence keeps old shape.
        if len(units) == 1:
            meta = classify_independently(units[0])
            return {
                'mode': 'classify-single',
                'query': units[0],
                'instruction': units[0],
                'category': meta['category'],
                'source': meta['source'],
                'model_confidence': meta['model_confidence'],
                'top2_gap': meta.get('top2_gap', 0.0),
                'label_probabilities': meta.get('probabilities', {}),
            }

        results = []
        for i, unit in enumerate(units, 1):
            meta = classify_independently(unit)
            results.append({
                'step': i,
                'instruction': unit,
                'category': meta['category'],
                'source': meta['source'],
                'model_confidence': meta['model_confidence'],
                'top2_gap': meta.get('top2_gap', 0.0),
                'label_probabilities': meta.get('probabilities', {}),
            })

        return {
            'mode': 'classify-single',
            'query': instruction,
            'classifier': 'lora-model-only',
            'instructions': results,
        }

    return _previous_process_request(data)

print("✓ Raw classify now splits multi-sentence input and classifies each sentence separately")

In [ ]:
# Cell 7.5: Website mode quality override (high-fidelity extraction)
import re

def _normalize_query_for_relevance(query):
    q = (query or "").lower()
    replacements = {
        "cook": "make",
        "cooking": "making",
        "prepare": "make",
        "preparation": "recipe",
        "brew": "make",
        "tea recipe": "recipe",
    }
    for k, v in replacements.items():
        q = q.replace(k, v)
    return q

def _is_navigation_or_recipe_noise(text):
    t = re.sub(r"\s+", " ", str(text or "")).strip().lower()
    if not t:
        return True

    noise_patterns = [
        r"^related\s+step\s*by\s*step\s+recipes?\b",
        r"^related\s+recipes?\b",
        r"^step\s*by\s*step\s+(?:recipes?|guides?)\b",
        r"^you\s+may\s+also\s+like\b",
        r"^more\s+recipes?\b",
        r"^read\s+more\b",
        r"^watch\s+video\b",
        r"^print\s+recipe\b",
        r"^jump\s+to\s+recipe\b",
        r"^read\s*\|",
        r"^also\s+read\s*\|",
        r"\b(read|also read)\s*\|\b",
        r"^comments?\b",
        r"^advertisement\b",
        r"^share\b",
        r"step by step guide",
        r"guide here is a step",
    ]
    return any(re.search(p, t) for p in noise_patterns)

def _clean_extracted_step_text(text):
    s = re.sub(r"\s+", " ", str(text or "")).strip()
    if not s:
        return ""

    s = re.sub(r"(?i)\b(?:also\s+read|read)\s*\|.*$", "", s).strip()
    s = re.sub(r"(?i)^step\s*\d+\s*[:\-\.)]?\s*", "", s).strip()

    meta_patterns = [
        r"(?i)^congratulations!?\b",
        r"(?i)^do you know how to",
        r"(?i)^here is a step by step guide",
        r"(?i)^how to create a (?:gmail|google) account\??$",
        r"(?i)^how to \w+.*? step by step",
        r"(?i)^guide here is a",
    ]
    for p in meta_patterns:
        s = re.sub(p, "", s).strip()

    return s.strip(" .;,:-\"")

def _canonical_step_key(step):
    s = (step or "").lower()
    s = re.sub(r"\([^\)]*\)", " ", s)
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def _is_near_duplicate_key(a_key, b_key):
    if not a_key or not b_key:
        return False
    if a_key == b_key or a_key in b_key or b_key in a_key:
        return True

    a_tokens = set(a_key.split())
    b_tokens = set(b_key.split())
    if len(a_tokens) < 4 or len(b_tokens) < 4:
        return False

    inter = len(a_tokens & b_tokens)
    shorter = max(1, min(len(a_tokens), len(b_tokens)))
    overlap = inter / shorter
    return overlap >= 0.85 

def _dedup_preserve_order_preferring_longer(steps):
    cleaned = []
    for s in steps:
        s2 = re.sub(r"\s+", " ", str(s)).strip(" .")
        if s2:
            cleaned.append(s2)

    out = []
    keys = []
    for cand in cleaned:
        ckey = _canonical_step_key(cand)
        if not ckey:
            continue

        replaced = False
        skip = False

        for i, ok in enumerate(keys):
            if _is_near_duplicate_key(ckey, ok):
                if len(cand) > len(out[i]):
                    out[i] = cand
                    keys[i] = ckey
                    replaced = True
                else:
                    skip = True
                break

        if not replaced and not skip:
            out.append(cand)
            keys.append(ckey)

    return out

# =========================================================================
# STRICT GEMINI EXTRACTION PROMPT (Grammatically Shaped for LoRA)
# =========================================================================
def _gemini_strict_extraction(query, text):
    call_fn = globals().get('call_gemini')
    parse_fn = globals().get('parse_numbered_steps')
    if not call_fn or not parse_fn:
        return []
        
    prompt = f"""You are a strict procedural information extractor and summarizer.

User query: {query}

Web content:
{text[:25000]}

Task: Extract ONLY the actionable, procedural instructions relevant to the query from the content.
Strict Rules:
1) Order the steps chronologically.
2) Ignore all introductory text, author notes, generic descriptions, and serving ideas.
3) Output as a simple numbered list (1., 2., 3., ...).
4) FORMATTING REQUIREMENTS: You MUST rephrase the extracted steps to strongly match these specific grammatical styles, without changing the original meaning:
   - SINGLE ACTION: Keep it simple. Example: "Boil water", "Mash the dal".
   - SEQUENTIAL: If steps happen one after another, connect them with 'then'. Example: "Pour hot water then stir", "Roll the dough then heat the tawa".
   - PARALLEL: If combining multiple ingredients/actions, use 'and'. Example: "Add sugar and milk", "Add dal and jaggery".
   - EXCLUSIVE OBJECTS: If there's a choice of items, use 'or'. Example: "Use a spoon or a masher", "Use sugar or jaggery".
   - EXCLUSIVE ACTIONS: If there's a choice of verbs, use 'or'. Example: "Drink immediately or store in flask", "Flip or cook evenly".
   - PURPOSE: If an action has a goal, add 'if you want' or 'to'. Example: "Repeat the process if you want to cook evenly".
   - REASON/CONDITION: If an action depends on something, use 'If' or 'because'. Example: "If it does not fall, the mixture is thick".
5) Keep lines short (under 15 words) but ensure they fit the phrasing styles above.
6) Do not hallucinate. Use only facts from the text.
7) If no procedural steps are found, return exactly "NONE".
"""
    # Use low temperature for high fidelity factual extraction
    out = call_fn(prompt, temperature=0.1, max_output_tokens=1500)
    if not out or out.strip().upper() == "NONE":
        return []
    
    parsed = parse_fn(out)
    return _dedup_preserve_order_preferring_longer(parsed)


def process_website_content(content, query=None):
    print("Mode: WEBSITE content (Gemini-First Pipeline)")
    effective_query = _normalize_query_for_relevance(query or "Extract complete professional instructions from this webpage")

    raw_text = "" if content is None else str(content)
    total_words = len(re.findall(r"\S+", raw_text))
    print(f"   Raw payload size: chars={len(raw_text)} words={total_words}")
    _print_block("RAW WEB CONTENT RECEIVED (FULL)", raw_text, max_chars=max(200000, len(raw_text) + 100))
    _print_block("USER QUERY", effective_query, max_chars=1500)

    if not content or not str(content).strip():
        return {
            "mode": "website",
            "query": effective_query,
            "summary": "No usable content found on this page.",
            "classifier": "lora-model-only",
            "instructions": []
        }

    use_gemini = bool(globals().get("GEMINI_API_KEY"))
    actionable = []
    
    # 1. GEMINI FIRST: Always use Gemini if available to ensure clean, shaped steps
    if use_gemini:
        print("   Routing entire payload to Gemini for syntactical shaping & concise filtering...")
        actionable = _gemini_strict_extraction(effective_query, raw_text)
        
    # 2. FALLBACK ONLY: If Gemini is offline or fails, use a basic regex fallback
    if not actionable:
        print("   Gemini extraction skipped or failed. Falling back to basic text extraction...")
        lines = [x.strip() for x in raw_text.split("\n") if x.strip()]
        for line in lines:
            s = _clean_extracted_step_text(line)
            if 15 <= len(s) <= 500 and not _is_navigation_or_recipe_noise(s):
                actionable.append(s)
        actionable = _dedup_preserve_order_preferring_longer(actionable)[:30]

    _print_block(
        "FILTERED INSTRUCTIONS FROM WEB CONTENT (HIGH-FIDELITY)",
        "\n".join([f"{i+1}. {s}" for i, s in enumerate(actionable)]) if actionable else "NONE",
        max_chars=12000, 
    )

    if not actionable:
        return {
            "mode": "website",
            "query": effective_query,
            "summary": "Relevant text exists, but no stable procedural steps could be extracted from this page.",
            "classifier": "lora-model-only",
            "instructions": []
        }

    results = []
    for i, step in enumerate(actionable, 1):
        meta = classify_independently(step)
        results.append({
            "step": i,
            "instruction": step,
            "category": meta["category"],
            "input": step,
            "output": meta["category"],
            "source": meta["source"],
            "model_confidence": meta["model_confidence"],
            "top2_gap": meta.get("top2_gap", 0.0),
            "label_probabilities": meta.get("probabilities", {}),
        })
        print(f"   {i}. [{meta['category']}] conf={meta['model_confidence']} gap={meta.get('top2_gap', 0.0)}")

    summary_text = " ".join([x["instruction"] for x in results[:2]])
    return {
        "mode": "website",
        "query": effective_query,
        "summary": summary_text,
        "classifier": "lora-model-only",
        "instructions": results,
    }

def _pick_best_website_payload(data):
    payload_keys = ["content", "website_content", "pageContent"]
    candidates = []
    for k in payload_keys:
        v = data.get(k)
        if v is None:
            continue
        s = str(v)
        if s.strip():
            candidates.append((k, s))

    if not candidates:
        return None, ""
    key, text = max(candidates, key=lambda x: len(x[1]))
    return key, text

def process_request(data):
    payload_data = dict(data or {})
    selected_key, selected_text = _pick_best_website_payload(payload_data)

    if selected_key:
        payload_data["website_content"] = selected_text

    query = payload_data.get('query') or payload_data.get('text')
    paragraph = payload_data.get('paragraph')
    website_content = payload_data.get('website_content') or payload_data.get('pageContent') or payload_data.get('content')
    instruction = payload_data.get('instruction') or payload_data.get('text') or payload_data.get('query')
    mode = (payload_data.get('mode') or 'auto').lower().strip()

    if mode == 'website':
        if website_content:
            return process_website_content(website_content, query=query)
        return {
            'mode': 'website',
            'query': query or 'Website Content Analysis',
            'summary': 'Website mode selected but no page content was received.',
            'classifier': 'lora-model-only',
            'instructions': []
        }

    if mode == 'generate':
        if query:
            return process_query(query)
        return {'error': 'No query provided for generate mode.'}

    if mode in ('classify-single', 'raw-classify', 'raw classify', 'classify_instruction'):
        if not instruction:
            return {'error': 'No instruction provided for classify-single mode.'}

        splitter = globals().get('_split_raw_classify_units')
        units = splitter(instruction) if callable(splitter) else [str(instruction).strip()]
        units = [u for u in units if str(u).strip()]
        if not units:
            return {'error': 'No valid instruction text found after splitting.'}

        if len(units) == 1:
            meta = classify_independently(units[0])
            return {
                'mode': 'classify-single',
                'query': units[0],
                'instruction': units[0],
                'category': meta['category'],
                'source': meta['source'],
                'model_confidence': meta['model_confidence'],
                'top2_gap': meta.get('top2_gap', 0.0),
                'label_probabilities': meta.get('probabilities', {}),
            }

        results = []
        for i, unit in enumerate(units, 1):
            meta = classify_independently(unit)
            results.append({
                'step': i,
                'instruction': unit,
                'category': meta['category'],
                'source': meta['source'],
                'model_confidence': meta['model_confidence'],
                'top2_gap': meta.get('top2_gap', 0.0),
                'label_probabilities': meta.get('probabilities', {}),
            })

        return {
            'mode': 'classify-single',
            'query': instruction,
            'classifier': 'lora-model-only',
            'instructions': results,
        }

    if mode == 'classify' and paragraph:
        return process_paragraph(paragraph)

    if mode == 'auto':
        if paragraph and not website_content:
            return process_paragraph(paragraph)
        if website_content or is_website_intent_query(query):
            if website_content:
                return process_website_content(website_content, query=query)
            return {
                'mode': 'website',
                'query': query or 'Website Content Analysis',
                'summary': 'Website mode requested but no page content was received.',
                'classifier': 'lora-model-only',
                'instructions': []
            }
        if query:
            return process_query(query)

    if query:
        return process_query(query)

    return {'error': 'No valid input provided. Send query, paragraph, instruction, or website content.'}

print("✓ Updated Gemini extraction prompt to rephrase outputs directly into the 7 expected syntactical styles.")

In [ ]:
# Cell 8: Configure ngrok using uploaded binary + uploaded yml
import os
import stat
import shutil
from pyngrok import ngrok, conf

NGROK_SOURCE_BIN_PATH = "/kaggle/input/datasets/ranjaysingh07/ngrok-file/ngrok"
NGROK_CONFIG_PATH = "/kaggle/input/datasets/ranjaysingh07/ngrok-yml/ngrok.yml"
NGROK_WORKING_BIN_PATH = "/kaggle/working/ngrok"

if not os.path.exists(NGROK_SOURCE_BIN_PATH):
    raise FileNotFoundError(f"ngrok binary not found: {NGROK_SOURCE_BIN_PATH}")
if not os.path.exists(NGROK_CONFIG_PATH):
    raise FileNotFoundError(f"ngrok config file not found: {NGROK_CONFIG_PATH}")

# /kaggle/input is read-only, so copy binary to writable /kaggle/working
shutil.copy2(NGROK_SOURCE_BIN_PATH, NGROK_WORKING_BIN_PATH)

# Ensure copied binary is executable
os.chmod(NGROK_WORKING_BIN_PATH, os.stat(NGROK_WORKING_BIN_PATH).st_mode | stat.S_IEXEC)

# Build pyngrok config from uploaded yml + writable binary (no download required)
PYNGROK_CONFIG = conf.PyngrokConfig(
    ngrok_path=NGROK_WORKING_BIN_PATH,
    config_path=NGROK_CONFIG_PATH
)

print("ngrok configured from uploaded files")
print(f"source binary: {NGROK_SOURCE_BIN_PATH}")
print(f"working binary: {NGROK_WORKING_BIN_PATH}")
print(f"config: {NGROK_CONFIG_PATH}")

In [ ]:
# Cell 9: Flask Server with All Endpoints

from flask import Flask, request, jsonify
from pyngrok import ngrok, conf
import traceback
import re

app = Flask(__name__)

# CORS Helper

def cors_response(data, status=200):
    """Create response with CORS headers"""
    response = jsonify(data)
    response.headers['Access-Control-Allow-Origin'] = '*'
    response.headers['Access-Control-Allow-Headers'] = 'Content-Type'
    response.headers['Access-Control-Allow-Methods'] = 'GET, POST, OPTIONS'
    return response, status


def handle_options():
    """Handle OPTIONS preflight requests"""
    return cors_response({'status': 'ok'})


def split_instruction_units(text):
    """Split mixed raw text into sentence/list units for per-unit classification."""
    text = "" if text is None else str(text)
    text = re.sub(r'\s+', ' ', text).strip()
    if not text:
        return []

    numbered_parts = re.split(r'(?=\b\d{1,2}[\.)]\s+)', text)
    chunks = []
    if len([p for p in numbered_parts if p.strip()]) > 1:
        chunks = [re.sub(r'^\s*\d{1,2}[\.)]\s*', '', p).strip() for p in numbered_parts if p.strip()]
    else:
        chunks = [text]

    units = []
    for chunk in chunks:
        parts = re.split(r'(?<=[.!?])\s+|(?<=[.!?])(?=[A-Z])|\n+', chunk)
        for p in parts:
            s = re.sub(r'\s+', ' ', p).strip(' .;,:')
            if s:
                units.append(s)

    out = []
    seen = set()
    for u in units:
        key = u.lower()
        if key not in seen:
            seen.add(key)
            out.append(u)
    return out


# ENDPOINT 1: /parse - MAIN ENDPOINT (auto-detects mode)

@app.route('/parse', methods=['POST', 'OPTIONS'])
def parse():
    if request.method == 'OPTIONS':
        return handle_options()
    
    try:
        data = request.json or {}
        
        if not any([data.get('query'), data.get('text'), data.get('paragraph'), 
                    data.get('website_content'), data.get('pageContent')]):
            return cors_response({'error': 'No input provided'}, 400)
        
        result = process_request(data)
        return cors_response({'result': result})
    
    except Exception as e:
        traceback.print_exc()
        return cors_response({'error': str(e)}, 500)


# ENDPOINT 2: /generate - Generate instructions from query

@app.route('/generate', methods=['POST', 'OPTIONS'])
def generate():
    if request.method == 'OPTIONS':
        return handle_options()
    
    try:
        data = request.json or {}
        query = data.get('query') or data.get('text', '')
        
        if not query:
            return cors_response({'error': 'No query provided'}, 400)
        
        result = process_query(query)
        return cors_response({'result': result})
    
    except Exception as e:
        traceback.print_exc()
        return cors_response({'error': str(e)}, 500)

# ENDPOINT 3: /classify - Classify a paragraph

@app.route('/classify', methods=['POST', 'OPTIONS'])
def classify():
    if request.method == 'OPTIONS':
        return handle_options()
    
    try:
        data = request.json or {}
        paragraph = data.get('paragraph') or data.get('text', '')
        
        if not paragraph:
            return cors_response({'error': 'No paragraph provided'}, 400)
        
        result = process_paragraph(paragraph)
        return cors_response({'result': result})
    
    except Exception as e:
        traceback.print_exc()
        return cors_response({'error': str(e)}, 500)

# ENDPOINT 4: /website - Process website content

@app.route('/website', methods=['POST', 'OPTIONS'])
def website():
    if request.method == 'OPTIONS':
        return handle_options()
    
    try:
        data = request.json or {}
        content = data.get('content') or data.get('website_content') or data.get('pageContent') or data.get('text', '')
        query = data.get('query')
        
        if not content:
            return cors_response({'error': 'No content provided'}, 400)
        
        result = process_website_content(content, query=query)
        return cors_response({'result': result})
    
    except Exception as e:
        traceback.print_exc()
        return cors_response({'error': str(e)}, 500)

# ENDPOINT 5: /classify-single - Classify single or multi-sentence instruction

@app.route('/classify-single', methods=['POST', 'OPTIONS'])
def classify_single():
    if request.method == 'OPTIONS':
        return handle_options()
    
    try:
        data = request.json or {}
        instruction = data.get('instruction') or data.get('text', '')
        
        if not instruction:
            return cors_response({'error': 'No instruction provided'}, 400)

        units = split_instruction_units(instruction)
        if not units:
            return cors_response({'error': 'No valid instruction text found'}, 400)

        # Backward-compatible payload shape for one unit.
        if len(units) == 1:
            meta = classify_independently(units[0])
            return cors_response({
                'result': {
                    'instruction': units[0],
                    'category': meta['category'],
                    'source': meta['source'],
                    'model_confidence': meta['model_confidence'],
                    'top2_gap': meta.get('top2_gap', 0.0),
                    'label_probabilities': meta.get('probabilities', {}),
                }
            })

        results = []
        for i, unit in enumerate(units, 1):
            meta = classify_independently(unit)
            results.append({
                'step': i,
                'instruction': unit,
                'category': meta['category'],
                'source': meta['source'],
                'model_confidence': meta['model_confidence'],
                'top2_gap': meta.get('top2_gap', 0.0),
                'label_probabilities': meta.get('probabilities', {}),
            })

        return cors_response({
            'result': {
                'mode': 'classify-single',
                'classifier': 'lora-model-only',
                'instructions': results,
            }
        })
    
    except Exception as e:
        traceback.print_exc()
        return cors_response({'error': str(e)}, 500)

# ENDPOINT 6: /health - Health check

@app.route('/health', methods=['GET'])
def health():
    lora_status = "loaded" if adapter_loaded else "not loaded"
    return cors_response({
        'status': 'ok',
        'model': 'Llama 3.1 8B + LoRA (classification only)',
        'classification_model': CLASS_MODEL_ID,
        'generation_fallback_model': GEN_MODEL_ID,
        'lora_status': lora_status,
        'categories': CATEGORIES,
        'endpoints': ['/parse', '/generate', '/classify', '/website', '/classify-single', '/health']
    })[0]


# START SERVER

print("\n" + "="*70)
print("STARTING INSTRUCTION STRUCTURER SERVER")
print("="*70)

# Use config prepared in Cell 8
pyngrok_config = globals().get("PYNGROK_CONFIG")
if pyngrok_config is None:
    # Fallback config (expects writable ngrok binary path)
    pyngrok_config = conf.PyngrokConfig(
        ngrok_path="/kaggle/working/ngrok",
        config_path="/kaggle/input/datasets/ranjaysingh07/ngrok-yml/ngrok.yml"
    )

# Start ngrok tunnel
public_url = ngrok.connect(5000, pyngrok_config=pyngrok_config)
url = public_url.public_url

lora_msg = "LoRA LOADED" if adapter_loaded else "NO LoRA"

print(f"""SERVER IS RUNNING!
My SERVER URL (copy for Chrome extension): {url:<55}
Classification model: {CLASS_MODEL_ID}
LoRA status: {lora_msg}
""")

# Run the Flask server (this blocks)
app.run(host="0.0.0.0", port=5000)